In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
import pickle
import joblib

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics import median_absolute_error, r2_score

from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import KFold, RepeatedKFold, LeaveOneOut, GridSearchCV
from sklearn.model_selection import RandomizedSearchCV, cross_val_score

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import StandardScaler

import warnings

In [4]:
from tqdm.auto import tqdm
from sklearn import metrics
from sklearn.preprocessing import scale
from sklearn.metrics import mean_squared_error

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor

from scipy.stats import randint, uniform

In [5]:
df_train = pd.read_parquet('/content/drive/My Drive/Colab Notebooks/Dubai-Houses-MLOps/Data/processed/df_train.parquet')
df_valid = pd.read_parquet('/content/drive/My Drive/Colab Notebooks/Dubai-Houses-MLOps/Data/processed/df_valid.parquet')
df_test = pd.read_parquet('/content/drive/My Drive/Colab Notebooks/Dubai-Houses-MLOps/Data/processed/df_test.parquet')

In [6]:
# disable scientific notation for pandas
pd.set_option('display.float_format', lambda x: f'{x:.6f}')

import warnings
warnings.filterwarnings('ignore')

In [7]:
X_train = df_train.drop(['price', 'price_log1p'], axis = 1)

y_train = df_train['price_log1p']

In [8]:
cv3 = KFold(
    n_splits = 3,
    shuffle = True,
    random_state = 123
)

In [16]:
import joblib

model_xgb = joblib.load('/content/drive/My Drive/Colab Notebooks/Dubai-Houses-MLOps/models/model_xgb_gpu.pkl')
# model_cat = joblib.load('/content/drive/My Drive/Colab Notebooks/Dubai-Houses-MLOps/models/model_catboost_best-02.pkl')
model_lgbm = joblib.load('/content/drive/My Drive/Colab Notebooks/Dubai-Houses-MLOps/models/model_lgbm_best-02.pkl')
model_bag = joblib.load('/content/drive/My Drive/Colab Notebooks/Dubai-Houses-MLOps/models/model_bagging_best-02.pkl')

In [17]:
from sklearn.ensemble import StackingRegressor
from lightgbm import LGBMRegressor
from sklearn.pipeline import Pipeline

stack_pipeline = Pipeline([
    ('model', StackingRegressor(
        estimators=[
            ('xgb', model_xgb),
            ('lgbm', model_lgbm),
            ('bag', model_bag),
        ],
        final_estimator=LGBMRegressor(
            n_estimators=500,
            learning_rate=0.03,
            num_leaves=64,
            max_depth=-1
        ),
        passthrough=True,
        cv=5,
        n_jobs=-1
    ))
])

In [18]:
stack_pipeline.fit(X_train, y_train)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006201 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2850
[LightGBM] [Info] Number of data points in the train set: 24537, number of used features: 32
[LightGBM] [Info] Start training from score 14.540373


Pipeline(steps=[('model',
                 StackingRegressor(cv=5,
                                   estimators=[('xgb',
                                                Pipeline(steps=[('model',
                                                                 XGBRegressor(base_score=None,
                                                                              booster=None,
                                                                              callbacks=None,
                                                                              colsample_bylevel=np.float64(0.7574264031321645),
                                                                              colsample_bynode=np.float64(0.9955643169762546),
                                                                              colsample_bytree=np.float64(0.662787365394268),
                                                                              device=None,
                                                                              early_stopping_rounds=None,
                                                                              enable_categorical...
                                                                                                                  max_depth=15,
                                                                                                                  max_features='sqrt',
                                                                                                                  min_samples_leaf=7,
                                                                                                                  min_samples_split=21,
                                                                                                                  random_state=124),
                                                                                  max_features=np.float64(0.91330394402606),
                                                                                  max_samples=np.float64(0.7068643100658973),
                                                                                  n_estimators=277,
                                                                                  n_jobs=-1,
                                                                                  oob_score=True,
                                                                                  random_state=124))]))],
                                   final_estimator=LGBMRegressor(learning_rate=0.03,
                                                                 n_estimators=500,
                                                                 num_leaves=64),
                                   n_jobs=-1, passthrough=True))])

In [19]:
joblib.dump(
    stack_pipeline,
    '/content/drive/My Drive/Colab Notebooks/Dubai-Houses-MLOps/models/model_stacking_best-v2.pkl'
)


['/content/drive/My Drive/Colab Notebooks/Dubai-Houses-MLOps/models/model_stacking_best-v2.pkl']

In [23]:
import mlflow

mlflow.set_tracking_uri("file:///content/drive/My Drive/Colab Notebooks/Dubai-Houses-MLOps/mlruns")

In [24]:
with mlflow.start_run():
    mlflow.sklearn.log_model(
        sk_model=stack_pipeline,
        artifact_path="stacking_model"
    )

2026/05/09 19:55:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/09 19:55:47 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [26]:
import mlflow
import mlflow.sklearn

# save as MLflow format to Google Drive
mlflow.sklearn.save_model(
    sk_model=stack_pipeline,
    path='/content/drive/My Drive/Colab Notebooks/Dubai-Houses-MLOps/models/model_stacking_mlflow_gpu'
)

print("Saved as MLflow format!")

2026/05/09 19:57:18 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Saved as MLflow format!
